# Heavy Full — поэтапное продолжение HPO

Существующий поиск: `heavy_full_hpo_40ep_es_v2`. Готовые trials переиспользуются, незавершённые продолжаются из `last.pt`.

Сначала останови процессы старого поиска, установи новый `automated_hpo.py` и перезапусти kernel. Не изменяй общий пакет, пока из него работает другой HPO.

Запускай ячейки этапов по отдельности. По снимку архива этапы 1 и 2 завершены: после подготовки можно сразу выполнить **этап 3**. Более новые результаты на диске также будут учтены.


## 1. Пути, протокол и CHECKPOINTS

In [1]:
from pathlib import Path
import json

from satchange.automated_hpo import run_automated_parameter_search
from satchange.final_benchmark import run_final_benchmark


cwd = Path.cwd().resolve()
project_root = next(
    path for path in (cwd, *cwd.parents)
    if (path / "src/satchange").is_dir()
)
HPO_ROOT = project_root / "artifacts/hyperparameter_search/heavy/heavy_full"
SEARCH_NAME = "heavy_full_hpo_40ep_es_v2"
SEARCH_DIRECTORY = HPO_ROOT / SEARCH_NAME

# Use the experiment's actual settings, not a YAML edited after it started.
protocol = json.loads(
    (SEARCH_DIRECTORY / "automated_hpo_protocol.json").read_text(encoding="utf-8")
)
HPO_STAGES = protocol["stages"]

PROJECT_ROOT = Path(
    "."
)

# External initial checkpoints must remain the same for this experiment.
# Current incomplete trials use their own last.pt automatically.
CHECKPOINTS = [Path(item["path"]) for item in protocol.get("resume_from", [])]
GPU_IDS = [0, 1, 2, 3]  # Only GPUs allocated to this search.

print("Search:", SEARCH_DIRECTORY)
print("Stages:", [name for name, _ in HPO_STAGES])
print("External CHECKPOINTS:", len(CHECKPOINTS))
for path in CHECKPOINTS:
    print(path)


Search: artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2
Stages: ['01_batch_lr', '02_lr', '03_loss', '04_wd_scheduler']
External CHECKPOINTS: 0


## 2. Общие аргументы — выполнить один раз после каждого Restart Kernel

In [2]:
policy = protocol["early_stopping"] or {}

HPO_OPTIONS = dict(
    stages=HPO_STAGES,
    base_parameters=protocol["base"],
    model_name=protocol["model"],
    dataset_root=Path(protocol["dataset_root"]),
    thresholds=protocol["thresholds"],
    number_of_epochs=protocol["epochs"],
    patience=policy.get("patience"),
    min_epochs=policy.get("min_epochs", 1),
    min_delta=policy.get("min_delta", 0.0),
    output_root=HPO_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=GPU_IDS,
    checkpoint_interval=10,
    resume_incomplete=True,
    resume_from=CHECKPOINTS,
    f1_gap=protocol["gaps"][0],
    iou_gap=protocol["gaps"][1],
    seeds=protocol["seeds"] if protocol["evaluate_seeds"] else None,
    deployment_seed=protocol["deployment_seed"],
)

print("Maximum epochs:", HPO_OPTIONS["number_of_epochs"])
print("Image size:", HPO_OPTIONS["base_parameters"]["image_size"])
print("num_workers:", HPO_OPTIONS["base_parameters"]["num_workers"])
print("Early stopping:", policy)
print("Final seeds:", HPO_OPTIONS["seeds"])


Maximum epochs: 40
Image size: None
num_workers: 2
Early stopping: {'min_delta': 0.001, 'min_epochs': 20, 'monitor': 'validation_f1', 'patience': 12}
Final seeds: [42, 123, 777]


## Этап 1. Batch × learning rate

В архиве уже завершён. Эта ячейка прочитает результаты и пересчитает отбор без нового обучения.

In [3]:
stage1 = run_automated_parameter_search(**HPO_OPTIONS, stage="01_batch_lr")
print("Прошли дальше:", len(stage1["survivors"]))
print("Отбор:", stage1["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 10/12; best F1=0.830797, IoU=0.710566
Прошли дальше: 10
Отбор: artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_selection.json


## Этап 2. Уточнение learning rate

В архиве уже завершён. Эта ячейка прочитает результаты и пересчитает отбор без нового обучения.

In [4]:
stage2 = run_automated_parameter_search(**HPO_OPTIONS, stage="02_lr")
print("Прошли дальше:", len(stage2["survivors"]))
print("Отбор:", stage2["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 10/12; best F1=0.830797, IoU=0.710566
02_lr: new=0, reused=14
02_lr: keep 13/14; best F1=0.837355, IoU=0.720215
Прошли дальше: 13
Отбор: artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2/02_lr_selection.json


## Этап 3. Positive weight × BCE/Dice

Начни продолжение с этой ячейки после подготовки. Ранее законченные loss-trials не повторяются; для остальных используется одна очередь на выделенных GPU.

In [5]:
stage3 = run_automated_parameter_search(**HPO_OPTIONS, stage="03_loss")
print("Прошли дальше:", len(stage3["survivors"]))
print("Отбор:", stage3["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 10/12; best F1=0.830797, IoU=0.710566
02_lr: new=0, reused=14
02_lr: keep 13/14; best F1=0.837355, IoU=0.720215
03_loss: new=0, reused=52
03_loss: keep 33/52; best F1=0.853613, IoU=0.744612
Прошли дальше: 33
Отбор: artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2/03_loss_selection.json


## Этап 4. Weight decay × scheduler

Запусти после окончания этапа 3. Если он не завершён, функция сообщит об этом и не начнёт предыдущие этапы сама.

In [6]:
stage4 = run_automated_parameter_search(**HPO_OPTIONS, stage="04_wd_scheduler")
print("Прошли дальше:", len(stage4["survivors"]))
print("Отбор:", stage4["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 10/12; best F1=0.830797, IoU=0.710566
02_lr: new=0, reused=14
02_lr: keep 13/14; best F1=0.837355, IoU=0.720215
03_loss: new=0, reused=52
03_loss: keep 33/52; best F1=0.853613, IoU=0.744612
04_wd_scheduler: new=80, reused=52
04_wd_scheduler: queue=80, parallel_limit=4, gpu_ids=[0, 1, 2, 3]
Trials: 80
GPUs: [0, 1, 2, 3]
Single-GPU memory limit: 80000 MB

[START] Trial 000 -> GPU 3 | attempt=1
seed=42 | lr=0.0003 | wd=0.001 | pw=10 | BCE/Dice=0.5/0.5 | batch=1 | image_size=None | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=reduce_on_plateau
epochs=40

[START] Trial 001 -> GPU 0 | attempt=1
seed=42 | lr=0.0003 | wd=0.01 | pw=10 | BCE/Dice=0.5/0.5 | batch=1 | image_size=None | attention_dropout=0.0 | decoder_channels=128 | drop_path_

## Итоговая проверка и отчёт

Здесь отдельно запускается проверка оставшихся конфигураций на seeds `[42, 123, 777]` с прежним максимумом 40 эпох. Seed 42 переиспользуется при совпадении параметров. Это validation-проверка, не test и не отдельное более длинное финальное обучение.

In [7]:
summary = run_automated_parameter_search(**HPO_OPTIONS, stage="seeds")
best = summary["selected_by_mean_validation_f1"]
print("Validation F1:", best["mean_f1"], "+/-", best["std_f1"])
print("Validation IoU:", best["mean_iou"], "+/-", best["std_iou"])
print("Parameters:", best["parameters"])
print("Report:", SEARCH_DIRECTORY / "final_validation_summary.json")

01_batch_lr: new=0, reused=12
01_batch_lr: keep 10/12; best F1=0.830797, IoU=0.710566
02_lr: new=0, reused=14
02_lr: keep 13/14; best F1=0.837355, IoU=0.720215
03_loss: new=0, reused=52
03_loss: keep 33/52; best F1=0.853613, IoU=0.744612
04_wd_scheduler: new=0, reused=132
04_wd_scheduler: keep 78/132; best F1=0.863502, IoU=0.759793
seeds: new=156, reused=78
seeds: queue=156, parallel_limit=4, gpu_ids=[0, 1, 2, 3]
Trials: 156
GPUs: [0, 1, 2, 3]
Single-GPU memory limit: 80000 MB

[START] Trial 000 -> GPU 3 | attempt=1
seed=123 | lr=0.0001 | wd=0.01 | pw=10 | BCE/Dice=0.5/0.5 | batch=1 | image_size=None | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 0 | attempt=1
seed=777 | lr=0.0001 | wd=0.01 | pw=10 | BCE/Dice=0.5/0.5 | batch=1 | image_si


[Trial 000 | GPU 3] Epoch 1/40
Train loss:       0.806453
Validation loss:  0.653170
Best threshold:   0.65
Precision:        0.353696
Recall:           0.474237
F1:               0.405191
IoU:              0.254069

[Trial 001 | GPU 0] Epoch 1/40
Train loss:       0.832002
Validation loss:  0.770121
Best threshold:   0.85
Precision:        0.266708
Recall:           0.344890
F1:               0.300802
IoU:              0.177026

[Trial 002 | GPU 1] Epoch 1/40
Train loss:       0.806554
Validation loss:  0.699766
Best threshold:   0.60
Precision:        0.279469
Recall:           0.418051
F1:               0.334993
IoU:              0.201196

[Trial 003 | GPU 2] Epoch 1/40
Train loss:       0.831912
Validation loss:  0.749282
Best threshold:   0.85
Precision:        0.294381
Recall:           0.412279
F1:               0.343495
IoU:              0.207362

[Trial 000 | GPU 3] Epoch 2/40

[Trial 001 | GPU 0] Epoch 2/40
Train loss:       0.699925
Validation loss:  0.644788
Best threshold

In [ ]:
summary_path = checkpoint.parent / "summary.json"

if summary_path.is_file():
    trial_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    seed = trial_summary["parameters"]["seed"]
else:
    # Резервный вариант для собственного checkpoint без summary.json.
    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
    seed = saved["config"]["seed"]
    del saved


checkpoint = PROJECT_ROOT / (
    "artifacts/hyperparameter_search/heavy/heavy_resized/"
    "heavy_resized_from_best_60ep_v3/03_loss_4e5a461f5cf3/"
    "siam_mit_b3_diff/trial_007/best.pt"
)


summary = run_final_benchmark(
    variant="heavy_full",
    checkpoints={int(seed): checkpoint},

    project_root=PROJECT_ROOT,
    dataset_root=PROJECT_ROOT / "data/LEVIR-CD",

    device="auto",
    evaluation_batch_size=1,
    num_workers=0,
    precision="fp32",

    warmup=20,
    repetitions=100,

    strict_validation=True,
    overwrite=False,
)

print(summary["output_dir"])

## Альтернатива: продолжить всё автоматически

Вместо отдельных ячеек этапов можно вызвать прежний интерфейс:

```python
summary = run_automated_parameter_search(**HPO_OPTIONS)
```

Не запускай этот вызов параллельно с ячейкой этапа или вторым notebook того же поиска. `stage` не меняет протокол, поэтому переход между поэтапным и полным запуском не требует нового имени. Изменять сетку, workers, loss, budget или patience под тем же именем нельзя.

`CHECKPOINTS` — исходные **внешние** checkpoints эксперимента, а не список всех файлов на диске. Для Full список пустой, потому что поиск начинался с нуля. Для Resized в нём прежние четыре `best.pt`. Актуальные `last.pt` незавершённых trials обоих поисков контроллер находит через `requests/`; их не нужно добавлять в список или хешировать как неизменяемые внешние файлы.
